In [1]:
import os
os.listdir('/kaggle/input/datasets/abdullahehdayb/olist-abdullah-practice/')

['products.csv',
 'geolocation.csv',
 'order_items.csv',
 'ml_table_notebook1.csv',
 'orders.csv',
 'customers.csv',
 'sellers.csv',
 'product_category_name_translation.csv',
 'order_payments.csv',
 'order_reviews.csv']

In [2]:
import pandas as pd

DATA_DIR = '/kaggle/input/datasets/abdullahehdayb/olist-abdullah-practice/'
ml_table = pd.read_csv(DATA_DIR + 'ml_table_notebook1.csv', 
                         parse_dates=['order_delivered_customer_date', 'order_estimated_delivery_date'])

print(ml_table.shape)
ml_table.head()

(99441, 20)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,n_items,total_price,total_freight,n_distinct_products,n_distinct_sellers,total_payment,n_payment_entries,max_installments
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,29.99,8.72,1.0,1.0,38.71,3.0,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,118.70,22.76,1.0,1.0,141.46,1.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,159.90,19.22,1.0,1.0,179.12,1.0,3.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,45.00,27.20,1.0,1.0,72.20,1.0,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,19.90,8.72,1.0,1.0,28.62,1.0,1.0


In [3]:
print("Total orders:", len(ml_table))
print("Orders with a delivery date:", ml_table['order_delivered_customer_date'].notnull().sum())
print("Orders without a delivery date (excluded from labeling):", ml_table['order_delivered_customer_date'].isnull().sum())

# Check order_status breakdown for orders missing delivery date
ml_table[ml_table['order_delivered_customer_date'].isnull()]['order_status'].value_counts()

Total orders: 99441
Orders with a delivery date: 96476
Orders without a delivery date (excluded from labeling): 2965


order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [4]:
# Keep only orders that actually have a delivery date
labeled = ml_table[ml_table['order_delivered_customer_date'].notnull()].copy()

# Late = actual delivery happened after the estimated delivery date
labeled['is_late'] = (labeled['order_delivered_customer_date'] > labeled['order_estimated_delivery_date']).astype(int)

print("Labeled orders:", len(labeled))
print(labeled['is_late'].value_counts())
print(labeled['is_late'].value_counts(normalize=True))

Labeled orders: 96476
is_late
0    88649
1     7827
Name: count, dtype: int64
is_late
0    0.918871
1    0.081129
Name: proportion, dtype: float64


In [5]:
# Pick a few late orders and manually verify the dates make sense
late_sample = labeled[labeled['is_late'] == 1][['order_id', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'is_late']].head(5)
print("LATE examples:")
print(late_sample)

on_time_sample = labeled[labeled['is_late'] == 0][['order_id', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'is_late']].head(5)
print("\nON-TIME examples:")
print(on_time_sample)

LATE examples:
                            order_id order_delivered_customer_date  \
20  203096f03d82e0dffbc41ebc2e2bcfb7           2017-10-09 22:23:46   
25  fbf9ac61453ac646ce8ad9783d7d0af6           2018-03-21 22:03:54   
35  8563039e855156e48fccee4d611a3196           2018-03-20 00:59:25   
41  6ea2f835b4556291ffdc53fa0b3b95e8           2017-12-28 18:59:23   
57  66e4624ae69e7dc89bd50222b59f581f           2018-04-03 13:28:46   

   order_estimated_delivery_date  is_late  
20                    2017-09-28        1  
25                    2018-03-12        1  
35                    2018-03-20        1  
41                    2017-12-21        1  
57                    2018-04-02        1  

ON-TIME examples:
                           order_id order_delivered_customer_date  \
0  e481f51cbdc54678b7cc49136f2d6af7           2017-10-10 21:25:13   
1  53cdb2fc8bc7dce0b6741e2150273451           2018-08-07 15:27:45   
2  47770eb9100c2d0c44946d9cf07ec65d           2018-08-17 18:06:29   
3  94

In [6]:
labeled.to_csv('/kaggle/working/labeled_table_notebook2.csv', index=False)
print("Saved:", labeled.shape)

Saved: (96476, 21)
